# cjm-transcript-decomp-core

> A frontend-agnostic core for the transcript decomposition workflow — composes isolated capability workers (forced alignment, VAD, graph storage) into a headless pipeline that decomposes transcription run manifests into a VAD-aligned context-graph spine with traceable provenance, and a CLI as its first driver.

## Install

```bash
pip install cjm_transcript_decomp_core
```

## Project Structure

```
nbs/
├── alignment.ipynb # Pure forced-alignment logic (no capability calls): map FA words back to character spans
├── cli.ipynb       # The CLI driver — the decomposition core's first (and currently only) frontend.
├── graph.ipynb     # Graph-spine EXTENSION + skeptical-lens verification (stage 5, CR-18 revolution 2).
├── models.ipynb    # Lean data shapes for the transcript-decomposition pipeline: in-core mirrors of the
└── pipeline.ipynb  # The headless decomposition pipeline (stage 5: decomp is an EXTENDER). Load a
```

Total: 5 notebooks

## Module Dependencies

```mermaid
graph LR
    alignment["alignment<br/>alignment"]
    cli["cli<br/>cli"]
    graph_mod["graph<br/>graph"]
    models["models<br/>models"]
    pipeline["pipeline<br/>pipeline"]

    alignment --> models
    cli --> pipeline
    cli --> models
    graph_mod --> models
    pipeline --> models
    pipeline --> graph_mod
    pipeline --> alignment
```

*7 cross-module dependencies detected*

## CLI Reference

### `cjm-transcript-decomp-core` Command

```
usage: cjm-transcript-decomp-core [-h] {run} ...

Headless transcript decomposition: VAD + forced alignment -> fine-spine graph
extension.

positional arguments:
  {run}
    run       Extend a transcription-emitted graph root with the fine spine

options:
  -h, --help  show this help message and exit

```

For detailed help on any command, use `cjm-transcript-decomp-core <command> --help`.

## Module Overview

Detailed documentation for each module in the project:

### alignment (`alignment.ipynb`)
> Pure forced-alignment logic (no capability calls): map FA words back to character spans

#### Import

```python
from cjm_transcript_decomp_core.alignment import (
    map_fa_words_to_text,
    assign_words_to_chunks,
    build_segments_from_alignment,
    tier1_alignment_checks
)
```

#### Functions

```python
def _strip_punct(
    text: str,  # Text to normalize
) -> str:  # Text with punctuation removed
    "Strip punctuation from text for comparison with FA output."
```

```python
def map_fa_words_to_text(
    text: str,             # Original text with punctuation
    fa_items: List[FAWord],  # FA word-level alignment results
) -> List[Tuple[int, int]]:  # (start_char, end_char) spans into the original text
    """
    Map forced-alignment words back to character spans in the original text.
    
    Walks the original text, matching each FA word (punctuation-stripped) against
    original-text tokens; returns character offset pairs for each FA word.
    """
```

```python
def assign_words_to_chunks(
    fa_items: List[FAWord],     # FA word-level alignment results
    vad_chunks: List[VADChunk],  # VAD chunks with start/end times
) -> List[int]:  # Chunk index for each FA word
    """
    Assign each FA word to a VAD chunk by timestamp overlap.
    
    Words whose start_time falls within a chunk's [start, end] are assigned to
    that chunk; words in silence gaps go to the nearest chunk by time proximity.
    """
```

```python
def build_segments_from_alignment(
    text: str,                      # Original text with punctuation
    spans: List[Tuple[int, int]],   # Character spans from map_fa_words_to_text
    assignments: List[int],         # Chunk index per word from assign_words_to_chunks
    num_chunks: int,                # Total number of VAD chunks
    source_id: Optional[str] = None,           # Source row id for traceability
    source_provider_id: Optional[str] = None,  # Source provider identifier
) -> List[TextSegment]:  # One segment per VAD chunk
    """
    Build a TextSegment per VAD chunk by grouping words by chunk assignment.
    
    Each chunk's text is the original (punctuated) slice from the first to the
    last word assigned to it; chunks with no words become empty segments.
    """
```

```python
def tier1_alignment_checks(
    segments: List[TextSegment],  # Segments produced by build_segments_from_alignment
    vad_chunks: List[VADChunk],   # The VAD chunks they were aligned to
) -> List[str]:  # Human-readable warnings (empty = all clear)
    "Tier-1 deterministic pre-filters for the alignment-review seam (no AI)."
```

#### Variables

```python
_PUNCT_RE
```

### cli (`cli.ipynb`)
> The CLI driver — the decomposition core's first (and currently only) frontend.

#### Import

```python
from cjm_transcript_decomp_core.cli import (
    logger,
    build_parser,
    load_capabilities,
    run_command,
    main
)
```

#### Functions

```python
def build_parser() -> argparse.ArgumentParser:  # Configured CLI parser
    "Build the CLI parser (subcommands: run)."
```

```python
def load_capabilities(
    manager: CapabilityManager,   # Freshly constructed manager
    instance_ids: List[str],  # Capability names to load (default instances), in order
    configs: Optional[Dict[str, Dict[str, Any]]] = None,  # Per-capability config overrides (caller-wins, C8)
) -> None
    "Discover manifests + load each requested capability (default instance)."
```

```python
async def run_command(
    args: argparse.Namespace,  # Parsed CLI arguments for the `run` subcommand
) -> int:  # Process exit code (0 = all sources extended + verified)
    "Execute the `run` subcommand: extend a transcription-emitted root with the fine spine."
```

```python
def main(
    argv: Optional[List[str]] = None,  # Argument list override (None = sys.argv)
) -> int:  # Process exit code
    "CLI entry point (console script: `cjm-transcript-decomp-core`)."
```


### graph (`graph.ipynb`)
> Graph-spine EXTENSION + skeptical-lens verification (stage 5, CR-18 revolution 2).

#### Import

```python
from cjm_transcript_decomp_core.graph import (
    resolve_root_ids,
    build_extension_payload,
    SourceVerification,
    verify_source
)
```

#### Functions

```python
def resolve_root_ids(
    """
    Recompute the transcription-emitted root node ids from manifest data.
    
    Deterministic identity makes the extender lookup a RECOMPUTATION, not a
    search: Source = content hash; AudioSegment = (source, boundary range);
    AudioRendition = (audio segment, preprocessing chain) — the per-source
    `chain` ([] = raw convert-only) carried by the 0.3.0 manifest; Transcript =
    (rendition, transcriber, config_hash). No stored-id coupling between the
    cores. A pre-0.3.0 manifest (no `chain`) recomputes the raw rendition ids
    (empty chain), which is correct for any non-preprocessed run.
    """
```

```python
def build_extension_payload(
    source_entry: Dict[str, Any],             # One source entry from the transcription manifest
    capabilities_info: Dict[str, Dict[str, Any]],  # The transcription manifest's capabilities block
    vad_config_hash: str,                     # THIS run's VAD capability config hash (skeleton identity input)
    text_from: str,                           # Authoritative transcriber (layer-0 text designation)
    segments: List[DecompSegment],            # Ordered aligned segments (per-transcriber variants attached)
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], Dict[str, Any]]:  # (nodes, edges, ids)
    """
    Build the fine-spine EXTENSION payload (pure; no capability calls).
    
    Segment identity is audio-side (audio RENDITION, VAD config, chunk range) —
    shared across transcribers by construction, distinct per rendition (vocals
    isolation can yield different VAD chunking than raw). Each Segment carries
    the audio `TimeSlice` ref into its rendition + one `CharSlice` ref per
    transcriber variant; `text_from` is recorded per segment as provenance
    (which Transcript the layer-0 text came from), never as global config. Edges
    come from `grouped_spine_edges`: PART_OF to the OWNING AudioRendition,
    STARTS_WITH per rendition, NEXT chained source-wide across coarse boundaries.
    """
```

```python
async def verify_source(
    queue: JobQueue,          # Started job queue
    graph_id: str,            # Graph-storage capability id
    source_id: str,           # Source node id to verify
    rendition_ids: List[str], # The AudioRendition ids the run extended (the fine spine is scoped to these)
) -> Optional[SourceVerification]:  # Result, or None if the Source is not found
    """
    Verify a Source's committed extension via server-side AGGREGATES (D13/D19).
    
    Two-layer verification for the Source-rooted schema: the coarse spine
    (Source → AudioSegments) is checked with id-list edge counts (rendition-
    independent); the fine spine hangs under the run's AudioRendition set, so it
    is scoped through the batched far-end constraint
    `RelationPredicate("PART_OF", node_ids=rendition_ids)` (the C17 batch shape) —
    never a whole-neighborhood materialization, and never mixing a sibling
    rendition's spine (raw vs vocals coexist under the same AudioSegments). One
    bounded projection pass (sources + rendition_id) yields missing-sources,
    distinct locators, AND the per-rendition ownership set for the STARTS_WITH
    check.
    """
```

#### Classes

```python
@dataclass
class SourceVerification:
    """
    Skeptical-lens verification of one Source's fine-spine extension under a
    specific rendition set, computed by querying the graph directly (never
    trusting run state).
    """
    
    source_id: str  # Verified Source node id
    title: str  # Source title (read back from the graph)
    audio_segment_count: int  # AudioSegment nodes found under the Source (coarse spine; rendition-independent)
    rendition_count: int  # AudioRendition nodes the fine spine was scoped to (the run's extended renditions)
    segment_count: int  # Fine Segment nodes found under those renditions
    source_starts_with: bool  # Exactly 1 STARTS_WITH Source -> first AudioSegment
    aseg_next_complete: bool  # Coarse NEXT count == audio_segment_count - 1
    seg_next_complete: bool  # Fine NEXT count == segment_count - 1 (source-wide chain)
    part_of_complete: bool  # Fine PART_OF count == segment_count (Segment -> rendition)
    rendition_starts_with_complete: bool  # STARTS_WITH from renditions == #renditions owning >=1 Segment
    all_have_timing: bool  # Every Segment has start_time + end_time
    all_have_sources: bool  # Every Segment has >=1 SourceRef (the audio ref at minimum)
    source_locators: List[str] = field(...)  # Distinct provenance locator URIs
    
    def ok(self) -> bool:  # True when every structural check passes
        "All structural checks pass."
```


### models (`models.ipynb`)
> Lean data shapes for the transcript-decomposition pipeline: in-core mirrors of the

#### Import

```python
from cjm_transcript_decomp_core.models import (
    FAWord,
    VADChunk,
    TextSegment,
    SegmentVariant,
    DecompSegment,
    DecompConfig,
    DecompSourceRecord,
    DecompManifest,
    new_run_id
)
```

#### Functions

```python
def new_run_id() -> str:  # e.g. "decomp_20260607_153000_1a2b3c4d"
    "Generate a unique, sortable decomposition run id."
```

#### Classes

```python
@dataclass
class FAWord:
    """
    One word-level forced-alignment result (segment-local times).
    
    Lean in-core mirror of the forced-alignment capability's `ForcedAlignItem`
    wire shape, so the core needs no interface-library dependency for a
    three-field DTO (pass-2 evidence: the typed FA result belongs to the
    forced-alignment adapter interface).
    """
    
    text: str  # The aligned word (punctuation typically stripped by the model)
    start_time: float  # Word start in seconds (relative to the segment audio)
    end_time: float  # Word end in seconds (relative to the segment audio)
    
    def from_wire(
            cls,
            d: Dict[str, Any],  # Capability wire dict
        ) -> "FAWord":  # Parsed word (extra keys ignored)
        "Build from a capability wire dict, tolerating extra keys."
```

```python
@dataclass
class VADChunk:
    "A voice-activity time range within one pipeline segment (segment-local)."
    
    index: int  # Chunk index within the pipeline segment
    start_time: float  # Start in seconds (relative to the segment audio)
    end_time: float  # End in seconds (relative to the segment audio)
    
    def duration(self) -> float:  # Chunk duration in seconds
        "Chunk duration."
```

```python
@dataclass
class TextSegment:
    """
    A text segment produced by alignment, before graph commit.
    
    Lean in-core mirror of the page-centric `TextSegment` (which imports FastHTML
    card-stack types); the core carries only the alignment-relevant fields
    (E11 precedent — keep core logic free of UI-adjacent deps).
    """
    
    index: int  # Position within its pipeline segment
    text: str  # Segment text content
    source_id: Optional[str]  # Upstream transcription row id (job_id)
    source_provider_id: Optional[str]  # Producing capability name
    start_char: Optional[int]  # Start char offset into the pipeline-segment text
    end_char: Optional[int]  # End char offset into the pipeline-segment text
```

```python
@dataclass
class SegmentVariant:
    """
    One transcriber's text + char range for one fine segment (stage 5).
    
    Becomes a `GraphNodeRef(Transcript) + CharSlice` provenance ref on the
    committed Segment node — text is stored ONCE per transcriber at the coarse
    Transcript node; fine-grained variants are SLICES, never duplicated text
    (Thread-1 slices-until-promoted).
    """
    
    transcriber: str  # Transcriber capability name
    text: str  # This transcriber's text for the chunk
    start_char: Optional[int]  # Char offset into that transcriber's pipeline-segment text
    end_char: Optional[int]  # End char offset
    
    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form
            """Serialize to a plain dict."""
            return asdict(self)
    
    
    @dataclass
    class DecompSegment
        "Serialize to a plain dict."
```

```python
@dataclass
class DecompSegment:
    """
    One fine spine segment (stage 5: shared audio-side skeleton + per-transcriber variants).
    
    Identity is AUDIO-SIDE (owning pipeline segment + chunk range) — shared
    across transcribers by construction; `text` is the AUTHORITATIVE
    (accuracy-transcriber) alignment, with every transcriber's chunk text
    riding `variants` (the authoritative one included).
    """
    
    index: int  # Source-wide 0-based fine-spine position
    text: str  # Authoritative text (the --text-from transcriber's alignment; "" = no aligned words)
    start_time: float  # Start in source-audio seconds (navigation)
    end_time: float  # End in source-audio seconds
    chunk_start: float  # VAD chunk start (chunk-local seconds within the pipeline-segment WAV)
    chunk_end: float  # VAD chunk end (chunk-local seconds)
    vad_chunk_index: int  # Chunk index within its pipeline segment
    pseg_index: int  # Owning pipeline-segment (AudioSegment) index within the source
    variants: List[SegmentVariant] = field(...)  # Per-transcriber chunk texts + char ranges
    
    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form
        "Serialize to a plain dict."
```

```python
@dataclass
class DecompConfig:
    "Configuration for one transcript-decomposition run."
    
    vad_capability: str = 'cjm-capability-silero-vad'  # VAD capability id
    fa_capability: str = 'cjm-capability-qwen3-forced-aligner'  # Forced-alignment capability id
    graph_capability: str = 'cjm-capability-graph-sqlite'  # Graph-storage capability id
    text_from: Optional[str]  # Authoritative transcriber for layer-0 text (None: sole transcriber; REQUIRED for multi-transcriber manifests)
    language: str = 'English'  # Forced-alignment language
    media_type: str = 'audio'  # Source media type
    force: bool = False  # Bypass capability-side caches (VAD + FA)
    assume_yes: bool = False  # Auto-accept HITL seams (headless mode)
    
    def to_dict(self) -> Dict[str, Any]:  # Plain-dict snapshot for the manifest
        "Serialize to a plain dict."
```

```python
@dataclass
class DecompSourceRecord:
    """
    Record of one Source whose fine spine this run committed (stage 5:
    `Document` dissolved into `Source` — decomp EXTENDS the transcription-
    emitted root rather than creating a document).
    """
    
    source_node_id: str  # Graph Source node id (deterministic; recomputable from the content hash)
    source_path: str  # Originating source audio path (from the upstream manifest)
    title: str  # Source display title
    segment_count: int  # Number of fine Segment nodes committed
    segment_ids: List[str] = field(...)  # Graph Segment node ids, in order
    
    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form
        "Serialize to a plain dict."
```

```python
@dataclass
class DecompManifest:
    """
    Durable record of one decomposition run (proto-bundle; see CR-20).
    
    Schema 0.2.0 (stage 5): `documents` became `sources` (Document dissolved
    into Source); entries carry the deterministic Source node id.
    """
    
    run_id: str  # Unique run identifier
    created_at: float  # Unix timestamp at run start
    config: Dict[str, Any]  # DecompConfig snapshot
    source_manifest: str  # Path to the consumed transcription run manifest
    source_format: str = ''  # Upstream manifest format tag (interchange contract)
    source_version: str = ''  # Upstream manifest schema version
    capabilities: Dict[str, Dict[str, Any]] = field(...)  # instance_id -> identity/db_path/config_hash
    sources: List[DecompSourceRecord] = field(...)  # Extended sources, input order
    FORMAT: str = field(...)  # Format tag
    VERSION: str = field(...)  # Schema version
    
    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form for JSON serialization
            """Serialize to a plain dict with nested sources."""
            return {
                "format": self.FORMAT,
        "Serialize to a plain dict with nested sources."
    
    def save(
            self,
            path: Union[str, Path],  # Destination JSON file (parent dirs created)
        ) -> Path:  # The written path
        "Write the manifest as pretty-printed JSON."
```


### pipeline (`pipeline.ipynb`)
> The headless decomposition pipeline (stage 5: decomp is an EXTENDER). Load a

#### Import

```python
from cjm_transcript_decomp_core.pipeline import (
    logger,
    submit_and_wait,
    load_source_manifest,
    vad_chunks_from_result,
    fa_words_from_result,
    build_alignment_composition,
    decompose_source,
    confirm_seam,
    collect_capability_info,
    run_decomp
)
```

#### Functions

```python
async def submit_and_wait(
    "Submit one capability job, wait for it, and return its result (raise on failure)."
```

```python
def load_source_manifest(
    path: str,  # Path to a transcription-core run manifest JSON
) -> Dict[str, Any]:  # Parsed manifest dict
    """
    Load + lightly validate a transcription-core run manifest.
    
    Consumed as untyped JSON — the format/version tags are the only interchange
    contract (no shared schema type across cores; manifest-as-interchange, CR-20).
    """
```

```python
def vad_chunks_from_result(
    result: VADResult,  # Typed VAD result (wire-decoded at the proxy)
) -> List[VADChunk]:  # Segment-local VAD chunks, sorted, re-indexed
    """
    Normalize a typed VAD result into segment-local VAD chunks.
    
    Pure normalizer (stage 3): the capability invocation moved into the
    per-source composition (`build_alignment_composition`); this folds one
    node's typed result. Per-segment VAD (not whole-source) is the validated
    decomp path — originally a browser Web-Audio constraint, kept on the
    merits (D11).
    """
```

```python
def fa_words_from_result(
    result: "ForcedAlignResult",  # Typed FA result (wire-decoded at the proxy)
) -> List[FAWord]:  # Word-level alignment results
    "Normalize a typed forced-alignment result into FA words (pure; stage 3)."
```

```python
def build_alignment_composition(
    seg_list: List[Dict[str, Any]],  # Pipeline-segment entries from the transcription manifest (0.2.0)
    vad_id: str,               # VAD capability instance id
    fa_id: str,                # Forced-alignment capability instance id
    transcribers: List[str],   # Transcriber names to align (manifest `transcripts` keys, stable order)
    force: bool = False,       # Per-call cache-bypass control flag
) -> Tuple[Composition, List[Dict[str, Any]]]:  # (composition, per-pseg meta rows)
    """
    Build the whole-source M×(VAD ∥ T×FA) composition (D8 fan-in, stage-5 variants).
    
    Each pipeline segment contributes one VAD node + one FA node PER TRANSCRIBER
    with non-empty text — VAD is audio-only (run once; the shared skeleton), FA
    maps each transcriber's text onto it. All nodes are independent; the host's
    alignment fold fans them in. Pipeline segments where EVERY transcriber's
    text is empty are recorded as skipped meta rows and contribute no nodes.
    
    Stage 8 (Option C): every node rides the TASK CHANNEL now — VAD on
    vad/detect_speech, FA on forced_alignment/align — and per-call `force` rides
    `control` (the adapters own caching). The native-routed path is gone.
    """
```

```python
async def decompose_source(
    queue: JobQueue,
    cfg: DecompConfig,         # Run configuration
    source: Dict[str, Any],    # One source entry from the transcription manifest
    source_index: int,         # Position of this source within the run
    transcribers: List[str],   # Transcriber names (manifest order)
    text_from: str,            # Authoritative transcriber (layer-0 text)
) -> Tuple[str, List[DecompSegment]]:  # (source_path, ordered aligned segments)
    """
    Decompose one source into aligned fine segments with per-transcriber variants.
    
    The VAD-chunk skeleton is computed ONCE per pipeline segment (audio-only);
    each transcriber's text is forced-aligned onto it independently, then the
    pure fold merges per chunk: the `text_from` transcriber's alignment becomes
    the authoritative text, every transcriber's chunk text + char range rides
    `variants` (slice refs at commit). C4's "same skeleton, different text"
    duplication is gone — agreement is stored once by construction.
    """
```

```python
def confirm_seam(
    seam: str,                 # Seam label, e.g. "alignment-review"
    summary_lines: List[str],  # What the operator is being asked to accept
    warnings: List[str],       # Tier-1 warnings (logged prominently)
    assume_yes: bool = False,  # Headless mode: accept without prompting
) -> bool:  # True = proceed, False = operator aborted
    """
    HITL approval seam in its cheapest viable form (log + optional CLI prompt).
    
    Per-seam HITL-assist annotation (5 fields):
      1. signal: per-document summary + Tier-1 warnings
      2. deterministic pre-filter: tier1_alignment_checks (no AI)
      3. modality-bridge candidate: spectrogram / word-overlay review (future Tier 2)
      4. authoritative verifier: re-align or re-transcribe-and-compare (future Tier 3)
      5. flywheel capture: accept/abort decisions logged; durable capture is a
         pass-2 seam-contract concern, not solved here
    
    NOTE: input() blocks the event loop — acceptable because seams sit between
    stages with no jobs in flight; the pass-2 seam contract needs an async shape.
    """
```

```python
def collect_capability_info(
    manager: CapabilityManager,   # Manager holding the loaded capabilities
    instance_ids: List[str],  # Instance ids to record
) -> Dict[str, Dict[str, Any]]:  # instance_id -> {name, version, db_path, config_hash}
    """
    Record capability identity + data-DB pointers for the run manifest (provenance).
    
    Stage 5: also records each capability's EFFECTIVE config hash (the same
    `compute_config_hash` the empirical store keys on) — the VAD config hash is
    a Segment identity input. `db_path` prefers the effective config over the
    manifest default (a --graph-db-path override must be what downstream cores
    resolve; the D19 lesson). Stage 6 (0.2.1): the EFFECTIVE config dict is
    recorded READABLY beside its hash (the I8 lesson).
    """
```

```python
def _journal_run_event(
    """
    Append a host-tier run event to the journal (CR-14 follow-up).
    
    The cores are the trusted host writer class: RUN_STARTED/RUN_FINISHED
    bracket the run (run manifest <-> journal linkage by run_id) and
    VERIFY_OUTCOME makes skeptical-lens results durable rows (I14). No-op
    when the manager has no journal store; append failures stay LOUD.
    """
```

```python
async def run_decomp(
    manager: CapabilityManager,        # Manager with VAD + FA + graph capabilities loaded
    queue: JobQueue,               # Started job queue
    cfg: DecompConfig,             # Run configuration
    source_manifest_path: str,     # Transcription run manifest to decompose
    run_id: Optional[str] = None,  # Override run id (default: generated)
    actor: Optional[str] = None,   # Who/what initiated (journal attribution; CLI default cli:<user>)
) -> DecompManifest:  # Manifest of the sources extended
    """
    Extend every source in a transcription run manifest with its fine spine.
    
    Stage 5 (decomp-as-extender): the graph ROOT must already exist — decomp
    recomputes the deterministic root ids from the manifest, verifies the
    Source node is present (loud error pointing at transcription emission
    otherwise), aligns per transcriber, and attaches the fine spine under the
    run's AudioRendition (PART_OF rendition) via the layer's idempotent
    `extend_graph` (re-runs verify-collide). The fold is declared as a
    `Derivation` event when segments were actually created.
    """
```
